# **Assignment #1**

**Submitted By:** Piyush L. Shrestha

**Date:** February 3, 2026

## **TASK 1: Log Parser with Regex & File I/O**

Write a script that reads multiple log files in a directory, extracts all log entries with severity
ERROR or WARNING, and outputs:

- A CSV file with columns: timestamp, log level, message
- A JSON summary of counts per severity level

Handle malformed lines gracefully using regex.

In [44]:
import csv, re, json

# READING THE FILE.
file_ref = open('../data/logs/app2.log', 'r')

# PATH TO SAVE THE OUTPUT.
file_path = "../output/logs.csv"

# PATTERN TO MATCH THE FORMAT: YYYY-MM-DD HH:MM:SS [LEVEL] MESSAGE_STRING
pattern = re.compile(r'^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) \[(INFO|WARNING|ERROR)\] (.+)$')

# COUNTER
counter = {
    'INFO': 0,
    'WARNING': 0,
    'ERROR': 0,
    'UNKNOWN': 0
} 

# WRITING ONTO FILE (ONE ROW AT A TIME).
with open(file_path, 'w', newline='') as csvfile:
    
    # HEADER OF THE CSV FILE.
    fields = ['timestamp', 'log level', 'message']
    writer = csv.DictWriter(csvfile, fieldnames=fields)
    writer.writeheader()

    for line in file_ref:
        line = line.strip()
        match = pattern.match(line)
        if match:
            ts, log, msg = match.groups()
            writer.writerow({
                fields[0]: ts,
                fields[1]: log,
                fields[2]: msg
            })
            counter[log] += 1
        else:
            counter['UNKNOWN'] += 1

# INITALLY A DICT.
print(counter, type(counter))

# CONVERTING INTO JSON. READS AS STR.
counter = json.dumps(counter)
print(counter, type(counter))

{'INFO': 16, 'WARNING': 10, 'ERROR': 8, 'UNKNOWN': 6} <class 'dict'>
{"INFO": 16, "WARNING": 10, "ERROR": 8, "UNKNOWN": 6} <class 'str'>


## **TASK 2: Structured Data Extraction & Transformation**

Given a text file containing mixed structured records (name, email, phone number), use
regex to parse valid entries and output:
- A deduplicated set of all emails
- A list of names with valid phone numbers only

Save results to a JSON file with schema { "emails": [...], "contacts": [...] }.

In [ ]:
import re, json

# READING FILE.
file_ref = open("../data/contacts.txt", "r")

# PATTERN TO CHECK THE FORMAT, AS WELL AS VALID EMAIL AND PHONE.
# VALID PHONE FORMATS: (XXX) XXX-XXX, XXX-XXX-XXX, XXX-XXXX
pattern = re.compile(
    r'^([A-Za-z]+),\s*([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}),\s*((?:\d{3}-\d{3}-\d{4}|\(\d{3}\)\s*\d{3}-\d{4}))$'
)

# TO STORE NAMES WITH VALID PHONE NUMBERS.
names = {}
# USING SET TO AVOID DUPLICATE EMAILS.
emails = set()
# USING NORMAL LIST AS CRITERIA FOR DUPLICATE PHONE NUMBERS IS NOT MENTIONED.
contacts = []

# READING & STORING VALID LINES.
for line in file_ref:
    state = pattern.match(line.strip())
    if state:
        name, email, phone = state.groups()
        names.setdefault(name, phone)
        emails.add(email)
        contacts.append(phone)

# NAMES WITH VALID CONTACTS
print("NAMES WITH VALID PHONE NUMBERS:\n")
for name, contact in names.items():
    print(name, ': ', contact)

# CREATING & WRITING INTO A JSON FILE.
with open("../output/data.json", "w") as file_w:
    
    # CREATING DATA VARIABLE TO MATCH THE SCHEMA.
    data = {
        # AS SETS ARE NOT JSON SERIALIZABLE, CONVERTING THEM INTO LISTS.
        'emails': list(emails),
        'contacts': contacts
    }
    
    # WRITING DATA INTO JSON FILE.
    file_w.write(json.dumps(data, indent=4))
    # json.dump(data, file_w, indent=2)


NAMES WITH VALID PHONE NUMBERS:

Alice :  123-456-7890
Charlie :  (555) 123-4567
Daisy :  111-222-3333


## **TASK 3: CSV Normalizer**

Write a script that reads a CSV where some fields are missing or malformed (e.g., number
fields as text). The script should:

- Detect and correct common data problems
- Fill missing numeric values with the column mean
- Save both clean CSV and a JSON report of data quality issues (counts of errors per column).

**Note:** Installing *text-to-number* package.

```python
pip install text-to-number
````

In [17]:
from text_to_number import text_to_number
print(text_to_number("Ninety "), type(text_to_number("One")))
print(text_to_number("Ninety"))
print(text_to_number("Ninety Nine"))
print(text_to_number("Ninehjhajh"))

90  <class 'str'>
90
99
Ninehjhajh


In [ ]:
import csv, json
from text_to_number import text_to_number

error_count = {
    'id': 0,
    'name': 0,
    'marks': 0
}

error_index = {
    'id': [],
    'name': [],
    'marks': []
}

# READING FILE.
with open("../data/students_raw.csv", "r") as file_r:
    
    error_log = {}
    marks = []
    valid = 0
    total = 0
    row = 1
    rows = []
    
    # READING LINES OF CSV.
    for line in csv.DictReader(file_r):
        
        # CHECKING ID, IF EMPTY SETTING THE ROW COUNT VALUE AS ID.
        try:
            id = int(line['id'].strip())
        except:
            id = row
            error_count['id'] += 1
            error_index['id'].append(row)
    
        # CHECKING NAME (IF EMPTY SETTING IT AS UNKOWN).
        name = line.get('name', '').strip().title()
        if not name:
            name = "Unknown"
            error_count['name'] += 1
            error_index['name'].append(row)

        # CHECKING MARKS, SETTING None IF NOT READABLE.
        try:
            mark = int(text_to_number(line.get("marks", ""))) 
            total += mark
            valid += 1
        except:
            mark = None
            error_count['marks'] += 1
            error_index['marks'].append(row)
            
        # UPDATING ROW COUNT.
        row += 1
        # UPDATING DATA VARIABLE.
        rows.append({
            'id': id,
            'name': name,
            'marks': mark
        })

    # MEAN OF VALID MARKS.
    avg = total//valid
    
    # UPDATING None TO MEAN VALUE.
    for r in rows:
        if r['marks'] is None:
            r['marks'] = avg

    # SAVING CSV.
    with open("../output/students.csv", "w") as file_w:
        fields = ["id", "name", "marks"]
        file_w = csv.DictWriter(file_w, fieldnames=fields)
        file_w.writeheader()
        file_w.writerows(rows)
        
    # SAVING JSON.
    with open("../output/error_log.json", "w") as file_w:
        data = {
            'error_index': error_index,
            'error_count': error_count
        }
        json.dump(data, file_w, indent=4)


## **TASK 4: Nested JSON Processor**

Given a deeply nested JSON file (user profiles with activity logs), write functions to:

- Extract all user IDs who have performed a specific action
- Count how many times each action occurred
- Output a sorted list of actions by frequency

You should use dictionary logic and comprehension.

In [59]:
import json

actions = {}
actionsPerUser = {}

def countActions(data):
    for activity in data:
        # COUNTS INDIVIDUAL ACTIONS
        actions[activity['action']] = actions.get(activity['action'], 0) + activity['count']
        # COUNTS USER BASED ACTIONS
        actionsPerUser[user['user_id']][activity['action']] = actionsPerUser[user['user_id']].get(activity['action'], 0) + activity['count']

def getSorted(data, level=1, desc = True):
    if level == 1:
        return dict(sorted(data.items(), key=lambda item: item[1], reverse = desc))
    else:
        # SORTS INNER VALUES
        return {id: dict(sorted(action.items(), key = lambda i: i[1], reverse=desc)) for id, action in sorted(data.items())}

def displayJSON(title, data, gap = 4):
    print("\n"+title+"\n")
    print(json.dumps((data), indent=gap))
    
with open('../data/users.json', 'r') as file_r:
    data = json.load(file_r)
    for user in data:
        actionsPerUser.setdefault(user['user_id'], {})
        countActions(user['profile']['activity'])
    
    displayJSON("ACTIONS PER USER", getSorted(actionsPerUser, 2, True))
    displayJSON("ACTION COUNT", getSorted(actions, True))


ACTIONS PER USER

{
    "1": {
        "logout": 83,
        "purchase": 62,
        "login": 55
    },
    "2": {
        "login": 102,
        "logout": 83,
        "purchase": 26
    },
    "3": {
        "login": 147,
        "logout": 74,
        "purchase": 59
    },
    "4": {
        "purchase": 107,
        "login": 63,
        "logout": 45
    },
    "5": {
        "login": 98,
        "purchase": 81,
        "logout": 30
    },
    "6": {
        "purchase": 113,
        "logout": 85,
        "login": 35
    },
    "7": {
        "login": 114,
        "logout": 54,
        "purchase": 31
    },
    "8": {
        "login": 112,
        "purchase": 95,
        "logout": 26
    },
    "9": {
        "purchase": 103,
        "login": 74,
        "logout": 50
    },
    "10": {
        "login": 102,
        "logout": 83,
        "purchase": 71
    },
    "11": {
        "login": 79,
        "logout": 66,
        "purchase": 50
    },
    "12": {
        "logout": 80,
        "pu

## **TASK 5. Regex Rule Validator**

Create a module with regex-based validation functions for:

- Username rules (start with a letter, no spaces, no trailing underscore)
- Password rules (min 10 chars, uppercase, digit, special char)
- Email validation
- Then write a CLI interface that reads user input CSV and validates each row’s credentials, outputting results to a new CSV.

In [3]:
import sys, csv
sys.path.append('sub_systems')
import user_validator as validator

with open('../data/credentials.csv', 'r') as file_r:
    data = csv.DictReader(file_r)
    for row in data:
        val = row['username']+', '+row['password']+', '+row['email']
        if validator.validate_user(val):
            print(f'{row['username']} has valid data.')
        else:
            print(f'{row['username']} has invalid data.')
    
    validator.view_logs()

alice1 has valid data.
1bob has invalid data.
charlie_ has invalid data.
Daisy has invalid data.

2026/13/02/15/26 09:13:51 validate_user('alice1, Password@123, alice@gmail.com') SUCCESS 'Valid CSV Format' 
2026/13/02/15/26 09:13:51 validate_username('alice1') SUCCESS 'Valid Username' 
2026/13/02/15/26 09:13:51 validate_password('Password@123') SUCCESS 'Valid Password' 
2026/13/02/15/26 09:13:51 validate_email('alice@gmail.com') SUCCESS 'Valid Email' 
2026/13/02/15/26 09:13:51 validate_user('alice1, Password@123, alice@gmail.com') SUCCESS 'Valid Inputs' 
2026/13/02/15/26 09:13:51 validate_user('1bob, short, bob@mail') SUCCESS 'Valid CSV Format' 
2026/13/02/15/26 09:13:51 validate_username('1bob') ERROR 'Invalid Username' 
2026/13/02/15/26 09:13:51 validate_user('1bob, short, bob@mail') ERROR 'Invalid Inputs' 
2026/13/02/15/26 09:13:51 validate_user('charlie_, ValidPass#99, charlie@yahoo.com') SUCCESS 'Valid CSV Format' 
2026/13/02/15/26 09:13:51 validate_username('charlie_') ERROR 'Inv

## **TASK 6: Inventory Change Tracker**

You have two CSV files representing inventory snapshots at times T1 and T2.
Write a script that:

- Reads both CSVs into dictionaries
- Outputs added, removed, and quantity-changed items

Results should be saved to JSON and printed in a human-friendly report.

In [34]:
# ASSUMING T1 IS THE PREVIOUS STOCK, AND T2 IS THE NEWER STOCK REPORT.

import csv, json

out = {}

with open('../data/inventory_t1.csv', 'r') as f1_r, open('../data/inventory_t2.csv', 'r') as f2_r:
    data_1 = csv.DictReader(f1_r)
    data_2 = csv.DictReader(f2_r)
    
    for d1, d2 in zip(data_1, data_2):
        if d1['item'] == d2['item']:
            diff = int(d1['quantity']) - int(d2['quantity'])
            state = 'Added' if diff < 0 else 'Removed'
            print(f'{d1['item'].title()} changed to {d2['item']} quantities, {state.lower()} {abs(diff) if abs(diff) > 0 else int(d2['quantity'])} items.')
            out.setdefault(d1['item'], {'quantity': int(d2['quantity']), 'state': 'Updated - '+state})
        else:
            print(f'{d1['item'].title()} has been removed.')
            out.setdefault(d1['item'], {'quantity': 0, 'state': 'Removed'})
            print(f'{d2['item'].title()} has been added, and has {d2['quantity']} quantities.')
            out.setdefault(d2['item'], {'quantity': int(d2['quantity']), 'state': 'Added'})
            
    print(json.dumps(out, indent=4))

Apple changed to apple quantities, removed 5 items.
Banana changed to banana quantities, removed 30 items.
Orange has been removed.
Grape has been added, and has 15 quantities.
{
    "apple": {
        "quantity": 45,
        "state": "Updated - Removed"
    },
    "banana": {
        "quantity": 30,
        "state": "Updated - Removed"
    },
    "orange": {
        "quantity": 0,
        "state": "Removed"
    },
    "grape": {
        "quantity": 15,
        "state": "Added"
    }
}
